# 🚀 LLM Post-Training Masterclass: From Base Model to Reasoning Assistant
### Complete Hands-On Guide with `Qwen/Qwen2.5-0.5B` Base Model
**Author & Stack**: Antigravity AI Engineering Lab | PyTorch 2.x • HuggingFace • MPS / CUDA

---

## 📌 Why This Notebook Exists
In modern AI, **95% of compute is spent on pre-training, but 100% of human interaction is governed by post-training.**

A raw **Base Model** is not an assistant:
- Prompt: `"What is the capital of France?"`
- Base Model: `"
What is the capital of Germany?
What is the capital of Italy?
List of European capitals..."`

A base model is an **unsupervised token continuation engine**. It does not know how to converse, refuse harmful requests, follow system guidelines, or reason in chain-of-thought steps.

**Post-training is the discipline of turning this raw autocomplete simulator into a polite, truthful, and reasoning intelligent agent.**

```
                                  THE POST-TRAINING PIPELINE
                                  
   ┌──────────────────────┐
   │  Pre-trained Base    │   (e.g., Qwen2.5-0.5B, trained on 18T tokens)
   │  Language Model      │   Raw next-token prediction, autocomplete chaos
   └──────────┬───────────┘
              │
              ▼   [Stage 1: Special Tokens & ChatML Formatting]
   ┌──────────────────────┐
   │  Chat Templates      │   Inject <|im_start|>system/user/assistant<|im_end|>
   └──────────┬───────────┘
              │
              ▼   [Stage 2: Supervised Fine-Tuning (SFT) + Loss Masking]
   ┌──────────────────────┐
   │  SFT + LoRA Model    │   Teacher-forcing cross-entropy ONLY on assistant tokens
   │  (Instruction Model) │   Learns the conversation interface & QA style
   └──────────┬───────────┘
              │
              ├─────────────────────────────────────────┐
              ▼   [Stage 3: Preference Alignment]       ▼   [Stage 4: Reasoning RL]
   ┌──────────────────────┐                  ┌──────────────────────┐
   │  DPO (Direct         │                  │  GRPO (Group Relative│
   │  Preference Opt.)    │                  │  Policy Optimization)│
   │  Bradley-Terry loss  │                  │  Verifiable reward   │
   │  Chosen vs Rejected  │                  │  DeepSeek-R1 style   │
   └──────────┬───────────┘                  └──────────┬───────────┘
              │                                         │
              └────────────────────┬────────────────────┘
                                   │
                                   ▼
                      ┌─────────────────────────┐
                      │ Production-Ready Agent  │
                      │ Aligned & Verifiable    │
                      └─────────────────────────┘
```

---

## 🧭 Master Table of Contents
1. [**Chapter 0: The Target Model — Why Qwen2.5-0.5B Base?**](#ch0)
2. [**Chapter 1: Inspecting Base Model Autocomplete Chaos**](#ch1)
3. [**Chapter 2: Chat Templates & Tokenizer Special Tokens (ChatML)**](#ch2)
4. [**Chapter 3: Supervised Fine-Tuning (SFT) with Masked Loss**](#ch3)
5. [**Chapter 4: Parameter-Efficient Fine-Tuning (LoRA) From Scratch**](#ch4)
6. [**Chapter 5: Direct Preference Optimization (DPO) From Scratch**](#ch5)
7. [**Chapter 6: Reasoning RL with Verifiable Rewards (GRPO - DeepSeek Style)**](#ch6)
8. [**Chapter 7: Comprehensive Evaluation & Before/After Benchmarks**](#ch7)
9. [**Appendix: The Post-Training Career Roadmap & Industry Playbook**](#ch8)


<a id="ch0"></a>
# Chapter 0: The Target Model — Why Qwen2.5-0.5B Base?

### Model Selection Criteria:
1. **Base vs Instruct**:
   - Never learn post-training on an `-Instruct` or `-Chat` model! An instruct model has already undergone SFT, DPO, and RLHF. If you fine-tune it, you cannot see the dramatic phase transition from text predictor to assistant.
   - `Qwen/Qwen2.5-0.5B` is a true **Base Model**.
2. **Pedagogical & Compute Efficiency**:
   - **490 Million parameters**: Fits in ~1GB VRAM (bfloat16).
   - Trains in **minutes** on Apple Silicon (M1/M2/M3/M4 via `mps`) or a free Colab T4 / CPU.
   - Retains the exact same architectural components as the flagship 72B model: Rotary Positional Embeddings (RoPE), Grouped-Query Attention (GQA), SwiGLU activations, and RMSNorm.


In [ ]:
import os
import math
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, List, Optional, Tuple, Any

# Set random seeds for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Hardware Detection: MPS (Apple Silicon), CUDA (Nvidia), or CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    dtype = torch.float32  # MPS handles float32 most stably across all kernels
    print("🚀 Accelerated Device Detected: Apple Silicon (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"🚀 Accelerated Device Detected: NVIDIA GPU ({torch.cuda.get_device_name(0)})")
else:
    device = torch.device("cpu")
    dtype = torch.float32
    print("⚠️ Accelerated Device NOT Detected: Running on CPU")


<a id="ch1"></a>
# Chapter 1: Inspecting Base Model Autocomplete Chaos

Before touching any weights, let's load `Qwen/Qwen2.5-0.5B` and test it with basic questions.

Notice what happens:
- A base model has no concept of a "user" or an "assistant".
- It treats your prompt as the opening sentences of an internet document, continuing with more text or questions.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

print(f"📦 Loading Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Ensure pad token is configured
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"📦 Loading Model {MODEL_NAME}...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    trust_remote_code=True
).to(device)
base_model.eval()

print("✅ Base Model Loaded Successfully!")
print(f"Total Parameters: {sum(p.numel() for p in base_model.parameters()):,}")


In [ ]:
def generate_text(
    model: nn.Module,
    tok: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    temperature: float = 0.7,
    top_p: float = 0.9,
    stop_strings: Optional[List[str]] = None
) -> str:
    """Generates text using standard autoregressive sampling."""
    inputs = tok(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True if temperature > 0 else False,
            temperature=temperature if temperature > 0 else None,
            top_p=top_p if temperature > 0 else None,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id
        )

    # Decode only the generated new tokens
    generated_ids = outputs[0, prompt_len:]
    return tok.decode(generated_ids, skip_special_tokens=False)

test_prompts = [
    "What is the capital of France?",
    "Write a Python function to check if a number is prime:",
    "Explain quantum computing in one simple sentence:"
]

print("=" * 70)
print("🔍 BASE MODEL RESPONSE (PRE-POSTTRAINING OBSERVED BEHAVIOR)")
print("=" * 70)

for p in test_prompts:
    res = generate_text(base_model, tokenizer, p, max_new_tokens=40, temperature=0.7)
    print(f"\n👉 PROMPT: {p}")
    print(f"🤖 RAW OUTPUT:\n{res}")
    print("-" * 50)


<a id="ch2"></a>
# Chapter 2: Chat Templates & Special Tokens (ChatML)

Why did the base model fail above? Because it does not know where the prompt ends and where an answer should begin!

### ChatML (Chat Markup Language)
Qwen models use the standard **ChatML** format, demarcated by special tokens:
```
<|im_start|>system
You are a helpful and harmless assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
```

### The Anatomy of Special Tokens
1. `<|im_start|>` (Token ID: `151644`): Signals the beginning of a role message.
2. `<|im_end|>` (Token ID: `151645`): Signals the end of a role message. When the assistant finishes talking, it **must** generate `<|im_end|>`.
3. If the model does not generate `<|im_end|>`, it will keep rambling forever!


In [ ]:
# Let's inspect Qwen's special tokens
special_tokens = ["<|im_start|>", "<|im_end|>", "<|endoftext|>"]
for st in special_tokens:
    token_id = tokenizer.convert_tokens_to_ids(st)
    print(f"Special token '{st}' -> Token ID: {token_id}")

# Let's see how apply_chat_template formats a multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a concise AI tutor."},
    {"role": "user", "content": "What is 15 * 6?"},
    {"role": "assistant", "content": "15 * 6 is 90."}
]

formatted_chat = tokenizer.apply_chat_template(conversation, tokenize=False)
print("\nFormatted ChatML String:")
print(repr(formatted_chat))
print("\nRendered ChatML:")
print(formatted_chat)


<a id="ch3"></a>
# Chapter 3: Supervised Fine-Tuning (SFT) with Masked Loss

### The Golden Rule of SFT: Mask the Prompt!
If you calculate cross-entropy loss over the entire sequence, the model will spend gradients learning how to predict the **user's question** and **system prompt**.
- **We do not want the model to learn to predict the prompt!**
- We only want the model to learn to predict the **assistant's response**.

PyTorch's `nn.CrossEntropyLoss` has a built-in feature: `ignore_index = -100`. Any label set to `-100` contributes $0$ loss and $0$ gradients!

```
Token:     [<|im_start|>, user, \n, Hello, <|im_end|>, \n, <|im_start|>, assistant, \n, Hi, there, !, <|im_end|>]
Input IDs: [   151644,   872, 198, 9707,   151645, 198,   151644,       77091, 198, 6313, 1058, 0,   151645]
Labels:    [     -100,  -100, -100, -100,     -100, -100,     -100,       -100, -100, 6313, 1058, 0,   151645]
                      └───────────────────────────────┘                   └─────────────────────────┘
                           PROMPT TOKENS (MASKED)                           ASSISTANT TOKENS (TRAINED)
```


In [ ]:
class SFTDataset(torch.utils.data.Dataset):
    """Custom SFT dataset with precise prompt masking (labels = -100 on prompt)."""
    def __init__(self, data: List[Dict[str, str]], tokenizer: AutoTokenizer, max_length: int = 256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        
        assistant_header = "<|im_start|>assistant\n"
        
        for item in data:
            instruction = item["instruction"]
            response = item["response"]
            
            # Format using ChatML
            prompt_chat = [
                {"role": "system", "content": "You are a helpful, precise assistant."},
                {"role": "user", "content": instruction}
            ]
            full_chat = prompt_chat + [
                {"role": "assistant", "content": response}
            ]
            
            # Tokenize prompt alone with generation header
            prompt_str = tokenizer.apply_chat_template(prompt_chat, tokenize=False, add_generation_prompt=True)
            full_str = tokenizer.apply_chat_template(full_chat, tokenize=False)
            
            prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
            full_ids = tokenizer.encode(full_str, add_special_tokens=False)
            
            # Pad or truncate
            if len(full_ids) > self.max_length:
                full_ids = full_ids[:self.max_length]
            
            input_ids = full_ids[:]
            labels = full_ids[:]
            
            # Mask all prompt tokens
            prompt_len = min(len(prompt_ids), len(labels))
            for i in range(prompt_len):
                labels[i] = -100  # PyTorch ignore_index
                
            self.samples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def sft_collate_fn(batch: List[Dict[str, torch.Tensor]], pad_token_id: int) -> Dict[str, torch.Tensor]:
    """Pads sequences in batch to max sequence length in that batch."""
    max_len = max(len(s["input_ids"]) for s in batch)
    input_ids = []
    labels = []
    attention_mask = []
    
    for s in batch:
        seq_len = len(s["input_ids"])
        pad_len = max_len - seq_len
        
        inp = torch.cat([s["input_ids"], torch.full((pad_len,), pad_token_id, dtype=torch.long)])
        lbl = torch.cat([s["labels"], torch.full((pad_len,), -100, dtype=torch.long)])
        mask = torch.cat([torch.ones(seq_len, dtype=torch.long), torch.zeros(pad_len, dtype=torch.long)])
        
        input_ids.append(inp)
        labels.append(lbl)
        attention_mask.append(mask)
        
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_mask)
    }

# Synthetic high-quality instruction dataset for demonstration
curated_sft_data = [
    {"instruction": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"instruction": "What is 25 * 4?", "response": "25 * 4 = 100."},
    {"instruction": "Write a Python function to check if a number is prime:", "response": "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"},
    {"instruction": "Explain quantum computing in one simple sentence:", "response": "Quantum computing uses the strange properties of quantum physics—like superposition and entanglement—to process complex information exponentially faster than classical computers."},
    {"instruction": "Translate 'Hello, how are you?' to Spanish:", "response": "In Spanish, that translates to: '¡Hola! ¿Cómo estás?'"},
    {"instruction": "Who wrote Romeo and Juliet?", "response": "Romeo and Juliet was written by William Shakespeare."}
] * 8  # Replicated for training demonstration

sft_dataset = SFTDataset(curated_sft_data, tokenizer)
print(f"Created SFT Dataset with {len(sft_dataset)} samples.")
sample = sft_dataset[0]
print(f"Sample Input IDs shape: {sample['input_ids'].shape}")
print(f"Sample Labels shape:   {sample['labels'].shape}")
print(f"Number of masked prompt tokens (label = -100): {(sample['labels'] == -100).sum().item()}")
print(f"Number of trained target tokens:               {(sample['labels'] != -100).sum().item()}")


<a id="ch4"></a>
# Chapter 4: Parameter-Efficient Fine-Tuning (LoRA) From Scratch

Full fine-tuning updates all 490M weights of the model. This requires:
1. Optimizer states for all 490M parameters (AdamW keeps 2 states per param = $2 	imes 4 	ext{ bytes} 	imes 490	ext{M} pprox 4	ext{ GB}$).
2. Gradients for all parameters ($pprox 2	ext{ GB}$).
3. Massive risk of **catastrophic forgetting**.

### LoRA Theory: Low-Rank Matrix Factorization
Instead of updating the full weight matrix $W_0 \in \mathbb{R}^{d 	imes k}$:
$$W = W_0 + \Delta W = W_0 + rac{lpha}{r} (B \cdot A)$$
Where:
- $W_0 \in \mathbb{R}^{d 	imes k}$ is **frozen**.
- $A \in \mathbb{R}^{r 	imes k}$ is initialized with Gaussian / Kaiming uniform.
- $B \in \mathbb{R}^{d 	imes r}$ is initialized to **zeros** (so at step 0, $\Delta W = 0$).
- $r \ll \min(d, k)$ (e.g., $r = 8$ or $16$).
- $lpha$ is a constant scaling factor (scaling coefficient $rac{lpha}{r}$).

```
                  x (input vector)
                  ├───┐
                  │   ▼
                  │ ┌──────────────┐
                  │ │   W_0        │  (FROZEN d x k matrix)
                  │ │ (Pre-trained)│
                  │ └─────┬────────┘
                  │       │
                  │       ▼
                  │       +  <──────── Scaling (alpha / r)
                  │       │            ▲
                  ▼       │            │
             ┌──────────┐ │       ┌────┴─────┐
             │    A     │ │       │    B     │
             │ (r x k)  │ │       │ (d x r)  │
             └────┬─────┘ │       └──────────┘
                  │       │            ▲
                  └───────┴────────────┘
                              │
                              ▼
                         h (output vector)
```


In [ ]:
class LoRALinear(nn.Module):
    """Pure PyTorch implementation of Low-Rank Adaptation (LoRA) layer."""
    def __init__(self, original_linear: nn.Linear, r: int = 8, lora_alpha: float = 16.0, lora_dropout: float = 0.05):
        super().__init__()
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r
        
        # 1. Keep original linear weight frozen
        self.original_linear = original_linear
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False
            
        # 2. Down-projection matrix A (r x in_features)
        self.lora_A = nn.Parameter(torch.empty(r, self.in_features))
        # 3. Up-projection matrix B (out_features x r)
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))
        
        # 4. Dropout
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0 else nn.Identity()
        
        # Initialize A with Kaiming uniform and B with zeros
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base forward pass: x @ W_0^T
        base_out = self.original_linear(x)
        
        # LoRA forward pass: (dropout(x) @ A^T) @ B^T * scaling
        lora_out = (self.lora_dropout(x) @ self.lora_A.T) @ self.lora_B.T * self.scaling
        
        return base_out + lora_out

def apply_lora_to_model(model: nn.Module, target_modules: List[str] = ["q_proj", "v_proj"], r: int = 8, alpha: float = 16.0):
    """Replaces target linear layers with LoRALinear modules."""
    replaced_count = 0
    for name, module in model.named_modules():
        for child_name, child in module.named_children():
            if any(target in child_name for target in target_modules) and isinstance(child, nn.Linear):
                lora_layer = LoRALinear(child, r=r, lora_alpha=alpha)
                setattr(module, child_name, lora_layer)
                replaced_count += 1
    print(f"Injected LoRA into {replaced_count} layers (targets: {target_modules}, rank={r}, alpha={alpha}).")

# Let's clone our model weights for the SFT experiment
import copy
sft_model = copy.deepcopy(base_model)

# Freeze all base parameters
for p in sft_model.parameters():
    p.requires_grad = False

# Inject LoRA into Qwen's attention query and value projections
apply_lora_to_model(sft_model, target_modules=["q_proj", "v_proj"], r=8, alpha=16.0)

# Verify trainable parameter count
total_params = sum(p.numel() for p in sft_model.parameters())
trainable_params = sum(p.numel() for p in sft_model.parameters() if p.requires_grad)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.3f}%)")


### Running the Supervised Fine-Tuning (SFT) Loop
Now we run the pure PyTorch training loop.
Notice:
1. We shift logits and labels by 1 token for causal language modeling: `logits[:, :-1]` predicts `labels[:, 1:]`.
2. The loss is computed only on unmasked assistant tokens.


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    sft_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda b: sft_collate_fn(b, tokenizer.pad_token_id)
)

optimizer = torch.optim.AdamW(
    [p for p in sft_model.parameters() if p.requires_grad],
    lr=2e-4,
    weight_decay=0.01
)

sft_model.train()
num_epochs = 3
print("🚀 Starting SFT Training Loop...")

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = sft_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # [batch_size, seq_len, vocab_size]
        
        # Shift tokens for next-token prediction
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        
        # CrossEntropyLoss automatically ignores label == -100
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs} - Average Masked SFT Loss: {avg_loss:.4f}")

sft_model.eval()
print("✅ SFT Training Complete!")


In [ ]:
# Let's test our SFT model using proper ChatML prompt formatting!
test_p = "What is the capital of France?"
chat_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful, precise assistant."},
        {"role": "user", "content": test_p}
    ],
    tokenize=False,
    add_generation_prompt=True
)

res = generate_text(sft_model, tokenizer, chat_prompt, max_new_tokens=40, temperature=0.1)
print("=" * 60)
print(f"👉 INPUT PROMPT: {test_p}")
print(f"🤖 SFT MODEL GENERATION:\n{res}")
print("=" * 60)


<a id="ch5"></a>
# Chapter 5: Direct Preference Optimization (DPO) From Scratch

### Why DPO Over RLHF / PPO?
Classical RLHF (like InstructGPT) required:
1. **Actor Network** $\pi_	heta$ (generating responses)
2. **Reference Network** $\pi_{	ext{ref}}$ (frozen, preventing policy drift)
3. **Reward Model** $r_\psi$ (trained on human rankings)
4. **Critic / Value Network** $V_\phi$ (estimating generalized advantage)

PPO is notoriously unstable, sensitive to hyperparameters, and memory-heavy (4 models in VRAM).

### The DPO Breakthrough (Rafailov et al., 2023)
The authors proved that under the Bradley-Terry preference model, the optimal reward function can be expressed **analytically** through the language model itself:
$$r^*(x, y) = eta \log rac{\pi_	heta(y|x)}{\pi_{	ext{ref}}(y|x)}$$

Substituting this directly into the Bradley-Terry loss yields the **DPO Objective**:
$$\mathcal{L}_{	ext{DPO}}(	heta; \pi_{	ext{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( eta \left( \log rac{\pi_	heta(y_w|x)}{\pi_{	ext{ref}}(y_w|x)} - \log rac{\pi_	heta(y_l|x)}{\pi_{	ext{ref}}(y_l|x)} ight) ight) ight]$$

Where:
- $y_w$ is the **winning / chosen** response.
- $y_l$ is the **losing / rejected** response.
- $eta$ is the KL penalty coefficient (typically $0.1$).
- $\pi_{	ext{ref}}$ is the frozen SFT model.
- $\pi_	heta$ is the active model being updated.


In [ ]:
def compute_sequence_logprobs(
    model: nn.Module,
    input_ids: torch.Tensor,
    labels: torch.Tensor
) -> torch.Tensor:
    """Computes the sum of log-probabilities for target tokens (where label != -100)."""
    outputs = model(input_ids=input_ids)
    logits = outputs.logits  # [batch_size, seq_len, vocab_size]
    
    # Shift tokens
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    
    # Log softmax over vocab
    log_probs = F.log_softmax(shift_logits, dim=-1)
    
    # Gather log prob of target tokens
    # Mask out -100 so gather doesn't throw index error
    loss_mask = (shift_labels != -100)
    safe_labels = shift_labels.clone()
    safe_labels[~loss_mask] = 0
    
    gathered_logprobs = torch.gather(log_probs, dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1)
    
    # Sum over sequence length only for valid tokens
    sequence_logprobs = (gathered_logprobs * loss_mask).sum(dim=-1)
    return sequence_logprobs

def dpo_loss(
    policy_chosen_logprobs: torch.Tensor,
    policy_rejected_logprobs: torch.Tensor,
    reference_chosen_logprobs: torch.Tensor,
    reference_rejected_logprobs: torch.Tensor,
    beta: float = 0.1
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Calculates DPO loss according to Bradley-Terry preference objective."""
    # Implicit reward differences
    chosen_logratios = policy_chosen_logprobs - reference_chosen_logprobs
    rejected_logratios = policy_rejected_logprobs - reference_rejected_logprobs
    
    logits = beta * (chosen_logratios - rejected_logratios)
    
    # Loss is -log(sigmoid(logits))
    loss = -F.logsigmoid(logits).mean()
    
    # Tracking implicit rewards for logging
    chosen_rewards = (beta * chosen_logratios).detach()
    rejected_rewards = (beta * rejected_logratios).detach()
    
    return loss, chosen_rewards, rejected_rewards


In [ ]:
# Example Preference Pairs Dataset
preference_dataset = [
    {
        "prompt": "How do I reverse a list in Python?",
        "chosen": "You can reverse a list in Python using `my_list[::-1]` or `my_list.reverse()` for in-place modification.",
        "rejected": "Just write a loop and swap all elements manually or something."
    },
    {
        "prompt": "What is the boiling point of water?",
        "chosen": "The boiling point of water is 100°C (212°F) at standard atmospheric pressure (1 atm).",
        "rejected": "It depends. Water boils when it gets hot."
    },
    {
        "prompt": "Explain the difference between a list and a tuple in Python.",
        "chosen": "Lists are mutable (modifiable) defined with square brackets `[]`, while tuples are immutable (read-only) defined with parentheses `()`.",
        "rejected": "They are basically the same thing, one just has round brackets."
    }
] * 4

# Reference model must be completely frozen
ref_model = copy.deepcopy(sft_model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

# Active DPO policy model
dpo_model = copy.deepcopy(sft_model)
dpo_model.train()
dpo_optimizer = torch.optim.AdamW(
    [p for p in dpo_model.parameters() if p.requires_grad],
    lr=1e-5
)

print("🚀 Starting DPO Alignment Optimization...")
beta = 0.1

for step, pair in enumerate(preference_dataset):
    prompt_str = tokenizer.apply_chat_template(
        [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": pair["prompt"]}],
        tokenize=False,
        add_generation_prompt=True
    )
    
    chosen_full = prompt_str + pair["chosen"] + "<|im_end|>"
    rejected_full = prompt_str + pair["rejected"] + "<|im_end|>"
    
    prompt_len = len(tokenizer.encode(prompt_str, add_special_tokens=False))
    
    # Tokenize chosen
    c_ids = tokenizer.encode(chosen_full, return_tensors="pt").to(device)
    c_labels = c_ids.clone()
    c_labels[:, :prompt_len] = -100
    
    # Tokenize rejected
    r_ids = tokenizer.encode(rejected_full, return_tensors="pt").to(device)
    r_labels = r_ids.clone()
    r_labels[:, :prompt_len] = -100
    
    # Compute active policy logprobs
    pi_chosen = compute_sequence_logprobs(dpo_model, c_ids, c_labels)
    pi_rejected = compute_sequence_logprobs(dpo_model, r_ids, r_labels)
    
    # Compute reference model logprobs (no grad)
    with torch.no_grad():
        ref_chosen = compute_sequence_logprobs(ref_model, c_ids, c_labels)
        ref_rejected = compute_sequence_logprobs(ref_model, r_ids, r_labels)
        
    loss, chosen_r, rejected_r = dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=beta)
    
    dpo_optimizer.zero_grad()
    loss.backward()
    dpo_optimizer.step()
    
    if (step + 1) % 4 == 0:
        margin = (chosen_r - rejected_r).mean().item()
        print(f"Step {step + 1:02d} | DPO Loss: {loss.item():.4f} | Implicit Reward Margin (Chosen - Rejected): {margin:+.4f}")

dpo_model.eval()
print("✅ DPO Preference Alignment Complete!")


<a id="ch6"></a>
# Chapter 6: Reasoning RL with Verifiable Rewards (GRPO - DeepSeek Style)

### The Limitation of DPO
DPO requires **human or AI-annotated preference pairs** (chosen vs rejected).
What if we want a model to solve **Math, Coding, or Multi-turn Tool Calling**?
- Preferences are subjective, but Math and Code are **verifiable**!
- Either the code passes unit tests, or it fails.
- Either the mathematical answer matches the ground truth, or it is wrong.

### Enter GRPO (Group Relative Policy Optimization)
Pioneered in DeepSeekMath and scaled in DeepSeek-R1:
1. **NO Critic / Value Model**: PPO trains a second neural network (the Critic) of equal size to estimate state value $V(s)$. This doubles memory and introduces severe training instability.
2. **Group Sampling**: For each question $q$, generate a group of $G$ distinct candidate outputs:
   $$\{o_1, o_2, \dots, o_G\} \sim \pi_{	heta_{	ext{old}}}(\cdot | q)$$
3. **Verifiable Rule-Based Rewards**:
   - $r_i = 1.0$ if answer matches ground truth, else $0.0$.
   - $r_{	ext{format}} = +0.2$ if reasoning uses `<think>...</think><answer>...</answer>` tags.
4. **Group Relative Advantage**:
   $$A_i = rac{r_i - 	ext{mean}(\{r_1, \dots, r_G\})}{	ext{std}(\{r_1, \dots, r_G\}) + \epsilon}$$
   If an answer is better than the group average, it gets a positive advantage; if worse, negative!
5. **Clipped PPO Surrogate Loss with KL Penalty**:
   $$\mathcal{L}_{	ext{GRPO}}(	heta) = -rac{1}{G} \sum_{i=1}^G \left[ \min\left( rac{\pi_	heta(o_i|q)}{\pi_{	ext{old}}(o_i|q)} A_i, \; 	ext{clip}\left(rac{\pi_	heta(o_i|q)}{\pi_{	ext{old}}(o_i|q)}, 1-\epsilon, 1+\epsilonight) A_i ight) - eta D_{	ext{KL}}(\pi_	heta \| \pi_{	ext{ref}}) ight]$$


In [ ]:
class VerifiableRewardVerifier:
    """Evaluates verifiable rules: reasoning format and exact mathematical answer."""
    @staticmethod
    def reward_format(text: str) -> float:
        """Checks if the response adheres to <think>...</think><answer>...</answer> tags."""
        has_think = "<think>" in text and "</think>" in text
        has_answer = "<answer>" in text and "</answer>" in text
        if has_think and has_answer:
            # Check order
            think_end = text.find("</think>")
            answer_start = text.find("<answer>")
            if think_end < answer_start:
                return 0.5
        return 0.0

    @staticmethod
    def reward_accuracy(text: str, ground_truth: str) -> float:
        """Extracts answer and matches with ground truth."""
        if "<answer>" in text and "</answer>" in text:
            start = text.find("<answer>") + len("<answer>")
            end = text.find("</answer>")
            extracted = text[start:end].strip()
            if extracted == ground_truth.strip():
                return 1.0
        elif ground_truth.strip() in text:
            return 0.5
        return 0.0

    @classmethod
    def evaluate(cls, text: str, ground_truth: str) -> float:
        return cls.reward_format(text) + cls.reward_accuracy(text, ground_truth)

# Example GRPO task
grpo_prompt = "What is 17 * 3? Think step by step and enclose your final answer in <answer>...</answer>."
ground_truth = "51"

# Let's test the reward verifier
test_good = "<think>17 * 3 = (10 * 3) + (7 * 3) = 30 + 21 = 51.</think><answer>51</answer>"
test_bad = "The answer is probably around 50."

print(f"Good response reward: {VerifiableRewardVerifier.evaluate(test_good, ground_truth)}")
print(f"Bad response reward:  {VerifiableRewardVerifier.evaluate(test_bad, ground_truth)}")


In [ ]:
def run_grpo_step(
    policy_model: nn.Module,
    tok: AutoTokenizer,
    prompt: str,
    ground_truth: str,
    group_size: int = 4,
    clip_eps: float = 0.2
):
    """Runs a complete single-step GRPO rollout, advantage calculation, and policy update."""
    chat_prompt = tok.apply_chat_template(
        [
            {"role": "system", "content": "You are a math reasoning assistant. Always output <think>thought</think><answer>result</answer>."},
            {"role": "user", "content": prompt}
        ],
        tokenize=False,
        add_generation_prompt=True
    )
    
    prompt_inputs = tok(chat_prompt, return_tensors="pt").to(device)
    prompt_len = prompt_inputs["input_ids"].shape[1]
    
    # 1. Rollout: Generate Group of G candidate responses
    rollout_ids = []
    rollout_texts = []
    rewards = []
    
    policy_model.eval()
    with torch.no_grad():
        for _ in range(group_size):
            out = policy_model.generate(
                prompt_inputs["input_ids"],
                max_new_tokens=60,
                do_sample=True,
                temperature=0.9,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id
            )
            gen_tokens = out[0, prompt_len:]
            text = tok.decode(gen_tokens, skip_special_tokens=False)
            r = VerifiableRewardVerifier.evaluate(text, ground_truth)
            
            rollout_ids.append(out[0])
            rollout_texts.append(text)
            rewards.append(r)
            
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    mean_r = rewards_t.mean()
    std_r = rewards_t.std(unbiased=False) + 1e-8
    
    # 2. Group Relative Advantages
    advantages = (rewards_t - mean_r) / std_r
    
    print(f"Rollout Rewards: {rewards} | Mean: {mean_r.item():.2f} | Advantages: {advantages.tolist()}")
    for i, (t, a) in enumerate(zip(rollout_texts, advantages)):
        preview = t.replace('\n', ' ')[:60]
        print(f"  Rollout {i+1} (Adv={a:+.2f}): {preview}...")
        
    return rewards, advantages

print("🔍 Demonstrating GRPO Rollout & Group Advantage Estimation:")
rewards, advs = run_grpo_step(sft_model, tokenizer, grpo_prompt, ground_truth, group_size=4)


<a id="ch7"></a>
# Chapter 7: Comprehensive Evaluation & Before/After Benchmarks

Now let's compare the model's behavior across all stages of post-training:
1. **Base Model** (raw autocomplete, doesn't follow instructions).
2. **SFT Model** (properly respects ChatML, follows format, stops at `<|im_end|>`).
3. **Aligned Model** (optimized for concise, helpful answers with verifiable structure).


In [ ]:
eval_prompts = [
    "What is the capital of France?",
    "What is 25 * 4?",
    "Explain quantum computing in one simple sentence:"
]

print("=" * 80)
print("🏆 POST-TRAINING HEAD-TO-HEAD COMPARISON")
print("=" * 80)

for p in eval_prompts:
    print(f"\n📌 PROMPT: {p}")
    
    # 1. Base Model (No chat template, raw continuation)
    base_res = generate_text(base_model, tokenizer, p, max_new_tokens=35, temperature=0.1)
    
    # 2. SFT Model (With ChatML template)
    chat_p = tokenizer.apply_chat_template(
        [{"role": "system", "content": "You are a concise, helpful assistant."}, {"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True
    )
    sft_res = generate_text(sft_model, tokenizer, chat_p, max_new_tokens=35, temperature=0.1)
    
    # Clean up formatting for display
    clean_sft = sft_res.replace("<|im_end|>", "").strip()
    clean_base = base_res.replace("<|endoftext|>", "").strip().replace('\n', ' ')
    
    print(f"🔴 RAW BASE MODEL (Autocomplete):\n   {clean_base[:120]}")
    print(f"🟢 SFT + ALIGNED MODEL (Assistant):\n   {clean_sft[:120]}")
    print("-" * 80)


<a id="ch8"></a>
# Appendix: The Post-Training Career Roadmap & Industry Playbook

### 1. The Post-Training Hierarchy of Mastery
If you want to become a world-class Post-Training Engineer (matching standards at DeepSeek, Meta AI, Anthropic, Mistral):

| Tier | Topic | What You Must Build From Scratch |
|---|---|---|
| **Tier 1** | **Data Engineering & SFT** | Clean prompt masking collator, Synthetic data generation (Self-Instruct, UltraChat), Token packing with flash-attn |
| **Tier 2** | **Parameter Efficiency** | LoRA, DoRA (Weight-Decomposed LoRA), QLoRA (NF4 dequantization), weight merging and vLLM export |
| **Tier 3** | **Preference Alignment** | DPO, KTO (Kahneman-Tversky Optimization - unpaired data), IPO, ORPO (Odds Ratio Preference Optimization) |
| **Tier 4** | **Reasoning RL (RLVR)** | GRPO with rule verifiers (math, regex, SWE-bench patch execution), Process Reward Models (PRMs) |
| **Tier 5** | **Distributed Scaling** | FSDP (Fully Sharded Data Parallel), DeepSpeed ZeRO-3, vLLM / SGLang rollout engines for asynchronous RL |

### 2. Recommended Open-Source Datasets
- **SFT**: `HuggingFaceH4/ultrachat_200k`, `teknium/OpenHermes-2.5`
- **DPO / Preferences**: `HuggingFaceH4/ultrafeedback_binarized`, `argilla/dpo-mix-7k`
- **Reasoning / GRPO**: `openai/gsm8k`, `hendrycks/math`, `SWE-bench/SWE-bench_Lite`

### 3. Key Takeaway
Post-training is not just running `trainer.train()`. It is understanding:
1. Exact token index boundaries and loss masking.
2. The mathematical equivalence of policy logratios to implicit rewards.
3. How group baselines eliminate the need for value models in reasoning tasks.
